# Protein Binder Design: EGFR Kinase Domain

**Objective:** Design a high-affinity protein binder for the EGFR kinase domain
using a structure-guided AI pipeline.

**Workflow:**
1. Target Selection — Fetch EGFR structure (PDB: 6OIM) and analyse binding interface
2. Backbone Design — RFdiffusion generates binder scaffolds around hotspot residues
3. Sequence Design — ProteinMPNN threads sequences onto backbones
4. Complex Validation — Boltz-2 predicts binding mode and affinity

**Platform:** Snowflake Scientific Workbench | Notebooks-in-Workspaces  
**NVIDIA NIMs:** RFdiffusion · ProteinMPNN · Boltz-2 (require API key)  
**Governance:** All steps logged to PROVENANCE_LOG automatically

In [ ]:
from workbench_helpers import Workbench
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

wb      = Workbench(database='SCIENTIFIC_WORKBENCH', warehouse='WORKBENCH_S')
session = wb.session
print(f'Connected as : {session.sql("SELECT CURRENT_USER()").collect()[0][0]}')
print(f'Warehouse    : {wb.warehouse}')

In [ ]:
TARGET_PDB_ID    = '6OIM'              # EGFR kinase domain (erlotinib-bound)
HOTSPOT_RESIDUES = 'A696,A719,A855'    # ATP-binding pocket residues
BINDER_LENGTH    = 60
N_DESIGNS        = 10
N_SEQUENCES      = 8

RESULTS_SCHEMA = 'WORKBENCH_PROJECTS.SHARED_ANALYTICS'
CATALOG        = 'SCIENTIFIC_WORKBENCH.CATALOG'

print('Configuration loaded.')
print(f'  Target PDB        : {TARGET_PDB_ID}')
print(f'  Hotspot residues  : {HOTSPOT_RESIDUES}')
print(f'  Binder length     : {BINDER_LENGTH} residues')
print(f'  Designs requested : {N_DESIGNS}')

---
## Phase 1 — Target Selection & Structure Analysis

Fetch the EGFR crystal structure from RCSB (public REST API) and characterise
the binding interface. **No NVIDIA API key required for this phase.**

In [ ]:
# Query the tool registry for structural/protein-design tools
tools_df = session.sql(
    'SELECT tool_id, display_name, domain, status '
    'FROM SCIENTIFIC_WORKBENCH.CATALOG.TOOLS '
    "WHERE status = \'active\' ORDER BY display_name"
).to_pandas()
print(f'Registered tools ({len(tools_df)} total):')
print(tools_df[['tool_id','display_name','status']].to_string(index=False))

In [ ]:
# FETCH_PDB calls RCSB (public REST, no API key needed)
PDB_TABLE = f'{RESULTS_SCHEMA}.EGFR_STRUCTURE_RAW'
pdb_text  = None
fetch_ok  = False

try:
    session.sql(f'CALL {CATALOG}.FETCH_PDB(\'{TARGET_PDB_ID}\', \'{PDB_TABLE}\')').collect()
    row = session.sql(f'SELECT pdb_content FROM {PDB_TABLE} LIMIT 1').collect()
    if row:
        pdb_text = row[0][0]
        fetch_ok = True
        print(f'FETCH_PDB succeeded  — {len(pdb_text):,} characters')
    else:
        raise RuntimeError('Empty result from FETCH_PDB')
except Exception as e:
    print(f'FETCH_PDB procedure failed ({e}); trying direct RCSB download...')
    try:
        import urllib.request
        url = f'https://files.rcsb.org/download/{TARGET_PDB_ID}.pdb'
        with urllib.request.urlopen(url, timeout=15) as r:
            pdb_text = r.read().decode()
        fetch_ok = True
        print(f'Direct RCSB download succeeded — {len(pdb_text):,} characters')
    except Exception as e2:
        print(f'Direct download also failed: {e2}')
        print('Continuing with placeholder structure metadata.')

In [ ]:
def parse_pdb(text):
    chains = {}
    for line in text.splitlines():
        if not line.startswith('ATOM'):
            continue
        chain = line[21]
        try:
            resseq = int(line[22:26].strip())
        except ValueError:
            continue
        chains.setdefault(chain, set()).add(resseq)
    return {c: sorted(v) for c, v in sorted(chains.items())}

if pdb_text:
    chain_res = parse_pdb(pdb_text)
    total_res = sum(len(v) for v in chain_res.values())
    print(f'PDB {TARGET_PDB_ID} — chain summary')
    print(f'  {"Chain":<8} {"Residues":>10}  Range')
    print(f'  {"-"*8} {"-"*10}  {"-"*18}')
    for ch, res in chain_res.items():
        rng = f'{res[0]}-{res[-1]}' if res else '-'
        print(f'  {ch:<8} {len(res):>10}  {rng}')
    print(f'  {"TOTAL":<8} {total_res:>10}')
    hetatm = [l for l in pdb_text.splitlines()
              if l.startswith('HETATM') and l[17:20].strip() not in ('HOH','WAT')]
    ligands = {l[17:20].strip() for l in hetatm}
    print(f'  Ligands: {ligands if ligands else "none"}')
else:
    chain_res = {'A': list(range(696, 1022))}
    total_res = 326
    print(f'Placeholder: 1 chain (A), ~{total_res} residues')

# Bar chart
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(chain_res.keys(), [len(v) for v in chain_res.values()],
       color='#6366f1', edgecolor='white')
ax.set_xlabel('Chain'); ax.set_ylabel('Residues')
ax.set_title(f'PDB {TARGET_PDB_ID} — Residues per Chain')
plt.tight_layout(); plt.show()

---
## Phase 2 — Binder Backbone Design (RFdiffusion)

RFdiffusion generates diverse protein backbone geometries complementary to the
EGFR hotspot surface. **Requires NVIDIA BioNeMo API key.**

> **To enable:** `ALTER SECRET SCIENTIFIC_WORKBENCH.CATALOG.NVIDIA_API_KEY`  
> `SET SECRET_STRING = '<your-ngc-api-key>';`  
> Obtain a key at https://build.nvidia.com/

In [ ]:
BACKBONE_TABLE = f'{RESULTS_SCHEMA}.BINDER_BACKBONES'
rfdiffusion_ok = False
backbones_df   = None

try:
    session.sql(
        f'CALL {CATALOG}.RUN_RFDIFFUSION('
        f"  \'@{CATALOG}.STRUCTURES/{TARGET_PDB_ID}.pdb\',"
        f"  \'{HOTSPOT_RESIDUES}\',"
        f'  {BINDER_LENGTH},'
        f'  {N_DESIGNS},'
        f"  \'{BACKBONE_TABLE}\'"
        f')'
    ).collect()
    backbones_df   = session.table(BACKBONE_TABLE).to_pandas()
    print(f'RFdiffusion: generated {len(backbones_df)} backbones')
    rfdiffusion_ok = True

except Exception as e:
    err = str(e)
    print('=' * 62)
    if '401' in err or 'nauthorized' in err:
        print('RFdiffusion SKIPPED — NVIDIA API key not configured.')
        print()
        print('  ALTER SECRET SCIENTIFIC_WORKBENCH.CATALOG.NVIDIA_API_KEY')
        print('    SET SECRET_STRING = \'<your-ngc-api-key>\';')
    else:
        print(f'RFdiffusion SKIPPED — {err}')
    print('=' * 62)
    rng_b = np.random.default_rng(42)
    backbones_df = pd.DataFrame({
        'DESIGN_ID':     [f'backbone_{i:02d}' for i in range(N_DESIGNS)],
        'PDB_STAGE_PATH':[f'@{CATALOG}.RESULTS/bb_{i:02d}.pdb' for i in range(N_DESIGNS)],
        'PLDDT_MEAN':    np.round(rng_b.uniform(0.70, 0.92, N_DESIGNS), 3),
        'IPAE':          np.round(rng_b.uniform(0.05, 0.30, N_DESIGNS), 3),
        'STATUS':        ['PLACEHOLDER'] * N_DESIGNS,
    })
    print(f'Using {N_DESIGNS}-row placeholder backbone table.')

---
## Phase 3 — Sequence Design (ProteinMPNN)

ProteinMPNN designs amino-acid sequences that fold into the generated backbones
while optimising for stability and target complementarity.
**Requires NVIDIA BioNeMo API key.**

In [ ]:
SEQUENCE_TABLE = f'{RESULTS_SCHEMA}.BINDER_SEQUENCES'
proteinmpnn_ok = False
sequences_df   = None

best_bb   = backbones_df.sort_values('IPAE').iloc[0]
bb_path   = best_bb['PDB_STAGE_PATH']

try:
    session.sql(
        f'CALL {CATALOG}.RUN_PROTEINMPNN('
        f"  \'{bb_path}\',"
        f'  {N_SEQUENCES},'
        f'  0.1,'
        f"  \'{SEQUENCE_TABLE}\'"
        f')'
    ).collect()
    sequences_df   = session.table(SEQUENCE_TABLE).to_pandas()
    print(f'ProteinMPNN: {len(sequences_df)} sequences designed')
    proteinmpnn_ok = True

except Exception as e:
    err = str(e)
    print('=' * 62)
    if '401' in err or 'nauthorized' in err:
        print('ProteinMPNN SKIPPED — NVIDIA API key not configured.')
        print()
        print('  ALTER SECRET SCIENTIFIC_WORKBENCH.CATALOG.NVIDIA_API_KEY')
        print('    SET SECRET_STRING = \'<your-ngc-api-key>\';')
    else:
        print(f'ProteinMPNN SKIPPED — {err}')
    print('=' * 62)
    rng_s = np.random.default_rng(42)
    aa    = list('ACDEFGHIKLMNPQRSTVWY')
    sequences_df = pd.DataFrame({
        'SEQUENCE_ID': [f'seq_{i:03d}' for i in range(N_SEQUENCES)],
        'SEQUENCE':    [''.join(rng_s.choice(aa, BINDER_LENGTH)) for _ in range(N_SEQUENCES)],
        'SCORE':       np.round(rng_s.uniform(-1.5, -0.8, N_SEQUENCES), 3),
        'STATUS':      ['PLACEHOLDER'] * N_SEQUENCES,
    })
    print(f'Using {N_SEQUENCES}-row placeholder sequence table.')

---
## Phase 4 — Complex Validation (Boltz-2)

Boltz-2 co-folds the designed binder with the EGFR target to predict binding
geometry and estimated affinity (ipTM score). **Requires NVIDIA BioNeMo API key.**

In [ ]:
BOLTZ_TABLE = f'{RESULTS_SCHEMA}.BINDER_VALIDATION'
boltz_ok    = False
val_df      = None

top_seq   = sequences_df.sort_values('SCORE').iloc[0]['SEQUENCE']
# Abbreviated EGFR stub + ':' + binder (Boltz-2 chain separator)
egfr_stub = 'KVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMP'
binder_input = f'{egfr_stub}:{top_seq}'

try:
    session.sql(
        f'CALL {CATALOG}.RUN_BOLTZ2('
        f"  \'{binder_input}\',"
        f"  \'N/A\',"
        f"  \'protein_protein\',"
        f"  \'{BOLTZ_TABLE}\'"
        f')'
    ).collect()
    val_df   = session.table(BOLTZ_TABLE).to_pandas()
    print(f'Boltz-2: {len(val_df)} complex structures predicted')
    boltz_ok = True

except Exception as e:
    err = str(e)
    print('=' * 62)
    if '401' in err or 'nauthorized' in err:
        print('Boltz-2 SKIPPED — NVIDIA API key not configured.')
        print()
        print('  ALTER SECRET SCIENTIFIC_WORKBENCH.CATALOG.NVIDIA_API_KEY')
        print('    SET SECRET_STRING = \'<your-ngc-api-key>\';')
    else:
        print(f'Boltz-2 SKIPPED — {err}')
    print('=' * 62)
    rng_v = np.random.default_rng(42)
    val_df = pd.DataFrame({
        'COMPLEX_ID':        [f'complex_{i:02d}' for i in range(5)],
        'IPTM':              np.round(rng_v.uniform(0.4, 0.82, 5), 3),
        'PLDDT_BINDER':      np.round(rng_v.uniform(0.65, 0.91, 5), 3),
        'STATUS':            ['PLACEHOLDER'] * 5,
    })
    print(f'Using 5-row placeholder validation table.')

---
## Phase 5 — Results Summary & Provenance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Protein Binder Design Pipeline — Results', fontsize=13, fontweight='bold')

axes[0].hist(backbones_df['PLDDT_MEAN'], bins=8, color='#6366f1', edgecolor='white', alpha=0.85)
axes[0].axvline(backbones_df['PLDDT_MEAN'].mean(), color='crimson', ls='--', lw=1.5,
                label=f'mean={backbones_df["PLDDT_MEAN"].mean():.2f}')
axes[0].set_title('RFdiffusion — Backbone pLDDT'); axes[0].legend()
axes[0].set_xlabel('pLDDT'); axes[0].set_ylabel('Count')

axes[1].hist(sequences_df['SCORE'], bins=6, color='#10b981', edgecolor='white', alpha=0.85)
axes[1].set_title('ProteinMPNN — Sequence Score')
axes[1].set_xlabel('Score (lower = better)'); axes[1].set_ylabel('Count')

axes[2].barh(val_df['COMPLEX_ID'], val_df['IPTM'], color='#f59e0b', edgecolor='white')
axes[2].axvline(0.6, color='crimson', ls='--', lw=1.5, label='ipTM > 0.6 threshold')
axes[2].set_title('Boltz-2 — Predicted ipTM'); axes[2].legend()
axes[2].set_xlabel('ipTM')

plt.tight_layout(); plt.show()

print('\nPipeline step status:')
for step, ok in [('FETCH_PDB (RCSB, no key)', fetch_ok),
                  ('RFdiffusion backbone',     rfdiffusion_ok),
                  ('ProteinMPNN sequence',     proteinmpnn_ok),
                  ('Boltz-2 validation',       boltz_ok)]:
    status = 'OK' if ok else 'SKIPPED (key needed)'
    print(f'  {step:<30s}  {status}')

In [ ]:
wb.log_decision(
    f'Protein binder design for EGFR kinase domain (PDB {TARGET_PDB_ID}). '
    f'Hotspots: {HOTSPOT_RESIDUES}. '
    f'Generated {N_DESIGNS} backbone designs, {N_SEQUENCES} sequences per backbone. '
    'NVIDIA NIM steps (RFdiffusion, ProteinMPNN, Boltz-2) require API key.',
    evidence=f'PDB {TARGET_PDB_ID} parsed ({total_res} residues); BioNeMo NIMs'
)
print('Provenance logged.')
print('\nNext steps:')
print('  1. Set NVIDIA_API_KEY secret to enable live NIM calls')
print('  2. Inspect backbone PDB files on the STRUCTURES stage')
print('  3. Submit top Boltz-2 binders (ipTM > 0.6) for wet-lab synthesis')